# **Práctica: Forecasting con SVM (SVR) e Integración con MLflow**

**Materia:** Aprendizaje Automático \
**Grupo:** N° 2 - Computación \
**Integrantes:** Leo Vásconez, Isaac Mora y Jean Pierre Valarezo \
**Profesor:** Ing. Remigio Hurtado \
**Fecha:** 16 de noviembre del 2026

**Objetivo:** Desarrollar un modelo de Máquinas de Soporte Vectorial para Regresión (SVR) con el dataset `retail_unificado.csv`, realizar *fine-tuning*, registrar los experimentos en MLflow y compararlos con el modelo de Red Neuronal Recurrente (RNN) previo.

In [ ]:
%pip install mlflow scikit-learn pandas numpy -q

### **1. Carga de librerías y configuración de MLflow**
Importamos las herramientas necesarias y definimos el experimento `Forecasting_SVR_Retail`, donde se almacenarán todas las corridas (parámetros, métricas y modelos).

In [39]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')  # Oculta advertencias de sklearn sobre feature_names
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Forecasting_SVR_Retail")

print("Imports funcionando correctamente y MLflow configurado")
print("URI actual:", mlflow.get_tracking_uri())

Imports funcionando correctamente y MLflow configurado
URI actual: http://127.0.0.1:5000


### **2. Preparación de datos y Feature Engineering**
Un SVR no procesa secuencias, por lo que convertimos la serie temporal en un problema tabular mediante **ventanas de tiempo (Lags)**: las ventas de los 7 días anteriores (`Lag_1` ... `Lag_7`) sirven para predecir la demanda del día actual. Los lags se calculan por producto para no mezclar historiales distintos.

In [40]:
LOOK_BACK = 7

# Ventas diarias por producto, ordenadas cronológicamente
df = pd.read_csv('retail_unificado.csv', parse_dates=['Fecha'])
ventas = (df.groupby(['Fecha', 'ID_Producto', 'Nombre_producto'])['Cantidad'].sum()
            .reset_index().sort_values(['ID_Producto', 'Fecha']))

# Lags: demanda de los últimos 7 días de cada producto
for i in range(1, LOOK_BACK + 1):
    ventas[f'Lag_{i}'] = ventas.groupby('ID_Producto')['Cantidad'].shift(i)
df_procesado = ventas.dropna()

print(f"Dataset procesado con {df_procesado.shape[0]} registros listos para SVR.")
display(df_procesado.head())

Dataset procesado con 2400 registros listos para SVR.


,Fecha,ID_Producto,Nombre_producto,Cantidad,Lag_1,Lag_2,Lag_3,Lag_4,Lag_5,Lag_6,Lag_7
188,2024-02-25,1,Leche,6,4.0,2.0,1.0,1.0,3.0,6.0,3.0
234,2024-03-03,1,Leche,2,6.0,4.0,2.0,1.0,1.0,3.0,6.0
246,2024-03-04,1,Leche,3,2.0,6.0,4.0,2.0,1.0,1.0,3.0
310,2024-03-13,1,Leche,3,3.0,2.0,6.0,4.0,2.0,1.0,1.0
319,2024-03-14,1,Leche,9,3.0,3.0,2.0,6.0,4.0,2.0,1.0


### **3. Escalado y división Train/Test**
SVM es muy sensible a la escala de los datos, por eso estandarizamos tanto los lags (`X`) como la cantidad (`y`). La división se hace **sin mezclar (`shuffle=False`)** para respetar el orden temporal: se entrena con el pasado y se prueba con el futuro.

In [41]:
cols_lag = [f'Lag_{i}' for i in range(1, LOOK_BACK + 1)]

scaler_X, scaler_y = StandardScaler(), StandardScaler()
X_scaled = scaler_X.fit_transform(df_procesado[cols_lag])
y_scaled = scaler_y.fit_transform(df_procesado[['Cantidad']]).ravel()

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, shuffle=False)

# Valores reales de prueba (sin escalar) y función de evaluación reutilizable
y_test_real = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()

def evaluar(modelo):
    """Predice sobre test, desescala y retorna (RMSE, MAE) en unidades reales."""
    preds = scaler_y.inverse_transform(modelo.predict(X_test).reshape(-1, 1)).ravel()
    return np.sqrt(mean_squared_error(y_test_real, preds)), mean_absolute_error(y_test_real, preds)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (1920, 7) | X_test: (480, 7)


### **4. Desarrollo y entrenamiento del modelo base (SVR)**
Entrenamos un SVR con kernel `rbf` e hiperparámetros por defecto. Este modelo es la referencia (*baseline*) y sus parámetros, métricas (RMSE y MAE) y el modelo quedan registrados en MLflow.

In [42]:
with mlflow.start_run(run_name="SVR_Base"):
    svr_base = SVR(kernel='rbf').fit(X_train, y_train)
    rmse_base, mae_base = evaluar(svr_base)

    mlflow.log_params({"kernel": "rbf", "tipo": "Base"})
    mlflow.log_metrics({"rmse": rmse_base, "mae": mae_base})
    mlflow.sklearn.log_model(svr_base, "modelo_svr_base")

    print("--- MODELO BASE ---")
    print(f"RMSE: {rmse_base:.2f}")
    print(f"MAE:  {mae_base:.2f}")

2026/11/16 15:45:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


--- MODELO BASE ---
RMSE: 2.20
MAE:  1.74
🏃 View run SVR_Base at: http://127.0.0.1:5000/#/experiments/1/runs/47bffd2a163f4aa8b32ac9c8d82f0b7d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### **5. Fine-tuning y optimización del modelo**
Con `GridSearchCV` probamos combinaciones de los hiperparámetros **C** (penalización), **gamma** (alcance de cada punto) y **kernel** (rbf o linear) usando validación cruzada de 3 pliegues, y nos quedamos con la que minimiza el error cuadrático medio.

In [43]:
with mlflow.start_run(run_name="SVR_Finetuning"):
    param_grid = {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto', 0.1, 0.01],
        'kernel': ['rbf', 'linear']
    }
    grid_search = GridSearchCV(SVR(), param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_svr = grid_search.best_estimator_
    rmse_opt, mae_opt = evaluar(best_svr)

    mlflow.log_params(grid_search.best_params_)
    mlflow.log_param("tipo", "Optimizado")
    mlflow.log_metrics({"rmse": rmse_opt, "mae": mae_opt})
    mlflow.sklearn.log_model(best_svr, "modelo_svr_optimizado")

    print("--- MODELO OPTIMIZADO (FINE-TUNING) ---")
    print(f"Mejores Hiperparámetros: {grid_search.best_params_}")
    print(f"RMSE Optimizado: {rmse_opt:.2f}")
    print(f"MAE Optimizado:  {mae_opt:.2f}")

2026/11/16 15:46:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


--- MODELO OPTIMIZADO (FINE-TUNING) ---
Mejores Hiperparámetros: {'C': 0.1, 'gamma': 0.01, 'kernel': 'rbf'}
RMSE Optimizado: 2.11
MAE Optimizado:  1.67
🏃 View run SVR_Finetuning at: http://127.0.0.1:5000/#/experiments/1/runs/499fdfa551ac43958c94d1c006b0b2dc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### **6. Consulta del historial en MLflow**
Recuperamos la tabla de corridas del experimento para comparar el modelo base con el optimizado sin salir del cuaderno.

In [44]:
experimento = mlflow.get_experiment_by_name("Forecasting_SVR_Retail")
runs_df = mlflow.search_runs(experiment_ids=[experimento.experiment_id])

display(runs_df[['tags.mlflow.runName', 'params.kernel', 'params.C', 'metrics.rmse', 'metrics.mae']])

,tags.mlflow.runName,params.kernel,params.C,metrics.rmse,metrics.mae
0,SVR_Finetuning,rbf,0.1,2.111188,1.668565
1,SVR_Base,rbf,NaN,2.195241,1.741819


### **7. Función de predicción de nuevo sample**
Definimos una función reutilizable que recibe el ID del producto y sus ventas de los últimos 7 días, y retorna la demanda esperada para el día siguiente usando el SVR optimizado. El proceso es: escalar la entrada, predecir, desescalar y asegurar que la demanda no sea negativa.

In [45]:
def predecir_demanda_nuevo_sample(id_producto, historico_7_dias):
    """Retorna la demanda predicha (unidades) para el día siguiente con el SVR optimizado."""
    if len(historico_7_dias) != LOOK_BACK:
        return f"Error: Se requiere un historial exacto de {LOOK_BACK} días."

    X_nuevo = scaler_X.transform(np.array(historico_7_dias).reshape(1, -1))
    pred_real = scaler_y.inverse_transform(best_svr.predict(X_nuevo).reshape(-1, 1))[0][0]
    demanda = max(0, int(round(pred_real)))  # La demanda no puede ser negativa

    print(f"[Predicción SVR] Producto ID {id_producto} | Demanda esperada: {demanda} unidades.")
    return demanda

### **Ejemplos de prueba de la función de predicción**
Probamos dos casos con comportamientos distintos: un producto de **alta rotación** (demanda constante y alta) y uno de **baja rotación** (demanda esporádica).

In [46]:
print("Caso 1: Producto de alta rotación (ej. Lácteos)")
predecir_demanda_nuevo_sample(id_producto=5, historico_7_dias=[15, 18, 20, 19, 22, 25, 20])

print("\nCaso 2: Producto de baja rotación (ej. Congelados)")
predecir_demanda_nuevo_sample(id_producto=25, historico_7_dias=[0, 2, 1, 0, 0, 3, 1])

Caso 1: Producto de alta rotación (ej. Lácteos)
[Predicción SVR] Producto ID 5 | Demanda esperada: 4 unidades.

Caso 2: Producto de baja rotación (ej. Congelados)
[Predicción SVR] Producto ID 25 | Demanda esperada: 3 unidades.


3

In [47]:
# Muestra los mejores hiperparámetros encontrados tras el fine-tuning
print(f"Mejores Hiperparámetros: {grid_search.best_params_}")

Mejores Hiperparámetros: {'C': 0.1, 'gamma': 0.01, 'kernel': 'rbf'}


### **8. Conclusiones y comparativa: SVR vs Red Neuronal Recurrente (RNN)**
Al analizar los resultados registrados en **MLflow** y compararlos con la RNN desarrollada en la Práctica 2, se concluye lo siguiente:

**1. Enfoque de modelado temporal**

• La **RNN (LSTM/GRU)** maneja las secuencias de forma nativa mediante estados ocultos, con entradas 3D `[muestras, pasos, características]`.

• El **SVR** exige transformar la serie en un dataset tabular con ventanas deslizantes (*Lags*). Es un preprocesamiento manual, pero el *Kernel Trick* le permite capturar relaciones no lineales de forma eficiente.

**2. Entrenamiento y fine-tuning**

• Optimizar $C$, $\gamma$ y el $kernel$ con *GridSearchCV* es más ágil que calibrar tasa de aprendizaje, épocas, *Early Stopping* y arquitectura (capas, Dropout) en la RNN.

• Las métricas de **MLflow** (RMSE y MAE) permiten verificar si el SVR optimizado mejora al SVR base, que sirve como *baseline*.

**3. Casos de uso**

• Para datasets minoristas pequeños o medianos, el SVR es competitivo, rápido y menos propenso al sobreajuste severo.

• Para historiales masivos, alta estacionalidad o predicciones a largo plazo, la RNN sigue siendo superior por su capacidad de abstraer patrones temporales complejos.

**Referencias:**

• Raschka, S. (2017). *Python Machine Learning* (2nd ed.). Packt Publishing.

• Documentación oficial de MLflow. (2024). *Tracking Experiments*. https://mlflow.org/docs/latest/tracking.html